In [19]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

**Note:** The project blueprint references column names like `MonthlyIncome`, `OverTime`, `JobSatisfaction`, `WorkLifeBalance`, and target `Attrition`. The actual dataset uses `MonthlySalary`, `OvertimeHoursPerMonth`, `WorkLifeBalanceScore`, and target `AttritionRisk`. All code below uses the real column names from the data.

In [20]:
PROCESSED_PATH = "../data/processed"
FEATURES_PATH = "../data/features"
os.makedirs(FEATURES_PATH, exist_ok=True)

emp = pd.read_csv(f"{PROCESSED_PATH}/employee_attrition_processed.csv")
print(f"Loaded: {emp.shape}")
print(f"Columns: {list(emp.columns)}")

Loaded: (500, 29)
Columns: ['EmployeeID', 'Name', 'Gender', 'Age', 'Department', 'JobRole', 'EducationLevel', 'JoiningDate', 'CountryCode', 'Country', 'PhoneNumber', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'LastLeaveDate', 'LeaveDayName', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalanceScore', 'PerformanceRating', 'AttritionRisk', 'CustomerSatisfaction_missing', 'MonthlySalary_outlier', 'OvertimeHoursPerMonth_outlier', 'ProjectsHandled_outlier', 'TrainingHours_outlier']


## 1. Drop Leaky and Useless Columns

Remove columns that would leak the answer or add no predictive value:
- **EmployeeID, Name, PhoneNumber**: Identifiers, not features
- **CountryCode**: Redundant with Country
- **JoiningDate**: Will derive features from this
- **LastLeaveDate**: If someone has a leave date, they likely already left (leaks target)

In [21]:
drop_cols = ["EmployeeID", "Name", "PhoneNumber", "CountryCode", 
             "JoiningDate", "LastLeaveDate", "PerformanceRating"]  # PerformanceRating: 50.63% importance = data leak

df = emp.drop(columns=drop_cols)
print(f"After dropping: {df.shape}")

After dropping: (500, 22)


## 2. Engineer Features

Each feature needs a business or statistical reason:
- **Income_per_year**: Salary relative to tenure — underpaid employees may leave
- **Gap_since_promotion**: Time without promotion — stagnation drives attrition
- **Satisfaction_score**: Combined wellbeing metric — captures overall employee sentiment
- **Experience_ratio**: Proportion of career at company — long-tenured employees are less likely to leave

In [22]:
# Income per year at company
# Reason: Employees who are underpaid relative to tenure may be more likely to leave
# Use epsilon to avoid division by zero for new employees (0 years)
epsilon = 0.1
df["Income_per_year"] = df["MonthlySalary"] / (df["YearsAtCompany"] + epsilon)

# Gap since last promotion
# Reason: not getting promotion for a long time can be a reason for attrition
current_year = pd.Timestamp.now().year
df["Gap_since_promotion"] = current_year - df["LastPromotionYear"]

# Combined satisfaction score
# Reason: Combines two wellbeing signals into one metric
# Use 0 for missing CustomerSatisfaction (pipeline will handle real imputation)
df["Satisfaction_score"] = (df["CustomerSatisfaction"].fillna(0) + df["WorkLifeBalanceScore"]) / 2

# Experience ratio (years at company / age)
# Reason: Proportion of career spent at company — higher ratio means more embedded
df["Experience_ratio"] = df["YearsAtCompany"] / df["Age"]

print("Engineered features:")
print(df[["Income_per_year", "Gap_since_promotion", "Satisfaction_score", "Experience_ratio"]].describe())

Engineered features:
       Income_per_year  Gap_since_promotion  Satisfaction_score  \
count       500.000000           500.000000          500.000000   
mean      17834.880422             5.582000            2.795830   
std       15579.285361             3.274251            2.085696   
min        2271.920530             2.000000           -1.415000   
25%        7675.416778             3.000000            1.185000   
50%       12459.466620             5.000000            2.715000   
75%       21956.080013             7.000000            4.008750   
max       85405.238095            16.000000            9.000000   

       Experience_ratio  
count        500.000000  
mean           0.218057  
std            0.137540  
min            0.033333  
25%            0.113529  
50%            0.192308  
75%            0.285714  
max            0.681818  


## 3. Handle Missing Values

CustomerSatisfaction is 63.8% missing. We already flagged this in Notebook 03 to preserve the signal.

**Important:** Median imputation will be done inside the model pipeline (not here) to avoid data leakage. The median must be computed on training data only, not the full dataset.

In [23]:
# Imputation is handled in the pipeline (Notebooks 06/07)
# This prevents data leakage from test data into training
print("Missing values will be imputed in the model pipeline (median for numeric, most_frequent for categorical)")

Missing values will be imputed in the model pipeline (median for numeric, most_frequent for categorical)


## 4. Identify Column Types

Identify numeric and categorical columns. Encoding will be done inside the model pipeline to avoid data leakage.

In [24]:
numeric_cols = ["MonthlySalary", "OvertimeHoursPerMonth", "LeavesTaken", "ProjectsHandled",
                "TrainingHours", "CustomerSatisfaction", "YearsAtCompany", "WorkLifeBalanceScore",
                "Age", "Income_per_year", "Gap_since_promotion",
                "Satisfaction_score", "Experience_ratio"]

categorical_cols = ["Gender", "Department", "JobRole", "Country", "LeaveDayName",
                    "EducationLevel", "LastPromotionYear"]

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

Numeric columns (13): ['MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'YearsAtCompany', 'WorkLifeBalanceScore', 'Age', 'Income_per_year', 'Gap_since_promotion', 'Satisfaction_score', 'Experience_ratio']
Categorical columns (7): ['Gender', 'Department', 'JobRole', 'Country', 'LeaveDayName', 'EducationLevel', 'LastPromotionYear']


In [25]:
# Save column metadata for Notebooks 06/07
import json
col_meta = {
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "target": "AttritionRisk"
}
with open(f"{FEATURES_PATH}/column_metadata.json", "w") as f:
    json.dump(col_meta, f, indent=2)
print("Saved column metadata")

Saved column metadata


## 5. Final Feature Set

Separate features (X) and target (y). Drop the target from features. **No encoding or imputation here** — that moves to the pipeline.

In [26]:
target = "AttritionRisk"
X = df.drop(columns=[target])
y = df[target]

print(f"Features: {X.shape}")
print(f"Target distribution: {y.value_counts().to_dict()}")
print(f"\nFeature columns: {list(X.columns)}")

Features: (500, 25)
Target distribution: {'No': 445, 'Yes': 55}

Feature columns: ['Gender', 'Age', 'Department', 'JobRole', 'EducationLevel', 'Country', 'MonthlySalary', 'OvertimeHoursPerMonth', 'LeavesTaken', 'LeaveDayName', 'ProjectsHandled', 'TrainingHours', 'CustomerSatisfaction', 'LastPromotionYear', 'YearsAtCompany', 'WorkLifeBalanceScore', 'CustomerSatisfaction_missing', 'MonthlySalary_outlier', 'OvertimeHoursPerMonth_outlier', 'ProjectsHandled_outlier', 'TrainingHours_outlier', 'Income_per_year', 'Gap_since_promotion', 'Satisfaction_score', 'Experience_ratio']


## 6. Save Feature Matrix

Save X and y as **raw features** (no imputation, no encoding). Notebooks 06/07 will handle preprocessing in their pipelines to prevent data leakage.

In [27]:
X.to_csv(f"{FEATURES_PATH}/X_features.csv", index=False)
y.to_csv(f"{FEATURES_PATH}/y_target.csv", index=False)

print(f"Saved X_features.csv: {X.shape}")
print(f"Saved y_target.csv: {y.shape}")

Saved X_features.csv: (500, 25)
Saved y_target.csv: (500,)
